This notebook is for cleaning the SemEval task datasets and doing feature engineering. For context, the span identification (task1_si) dataset is a binary identification of where the propaganda is, and the technique classification (task2_tc) dataset is a multi-class, human-annotated classification of which propaganda technique(s) were used in the span. We need si to distinguish between what is and isn't propaganda and tc to identify which technique was used.

In [1]:
import pandas as pd
import os
from pathlib import Path
from textblob import TextBlob
import re
import nltk
import json
import glob
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/frankiepike/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/frankiepike/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [2]:
#Define paths
BASE_DIR = Path("..").resolve()
INTERIM_DIR = BASE_DIR / "data" / "interim"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

In [3]:
#Load SemEval span identification data
semeval_si = pd.read_csv(INTERIM_DIR / "semeval_task1_si_merged.csv")
semeval_si.head()

,article_id,start_char,end_char,source_file,text_content
0,999001293,409,427,article999001293.task1-SI.labels,Top Florida County Election Official Illegally...
1,999001293,429,535,article999001293.task1-SI.labels,Top Florida County Election Official Illegally...
2,999001293,380,406,article999001293.task1-SI.labels,Top Florida County Election Official Illegally...
3,999001293,2307,2314,article999001293.task1-SI.labels,Top Florida County Election Official Illegally...
4,999001293,2849,2991,article999001293.task1-SI.labels,Top Florida County Election Official Illegally...


In [4]:
#Only show relevant text being evaluated for each row (using span on text)
semeval_si['span_text'] = semeval_si.apply(lambda row: row['text_content'][row['start_char']:row['end_char']], axis=1)
semeval_si.head()

,article_id,start_char,end_char,source_file,text_content,span_text
0,999001293,409,427,article999001293.task1-SI.labels,Top Florida County Election Official Illegally...,This is a disgrace
1,999001293,429,535,article999001293.task1-SI.labels,Top Florida County Election Official Illegally...,Have you ever heard of a close election in whi...
2,999001293,380,406,article999001293.task1-SI.labels,Top Florida County Election Official Illegally...,Every vote must be counted
3,999001293,2307,2314,article999001293.task1-SI.labels,Top Florida County Election Official Illegally...,outrage
4,999001293,2849,2991,article999001293.task1-SI.labels,Top Florida County Election Official Illegally...,Mayor Gillum conceded on Election Day and now ...


In [5]:
#Group spans by article because final model will only be given article/webpage full text, not spans
semeval_si_grouped = semeval_si.groupby(['article_id', 'text_content']).agg({
    'start_char': list,
    'end_char': list
}).reset_index()
semeval_si_grouped['propaganda_offsets'] = semeval_si_grouped.apply(
    lambda x: list(zip(x['start_char'], x['end_char'])), axis=1
)
semeval_si_grouped = semeval_si_grouped.rename(columns={'text_content': 'text'})
semeval_si_grouped = semeval_si_grouped.drop(columns=['start_char', 'end_char'])
semeval_si_grouped.head()

,article_id,text,propaganda_offsets
0,111111111,Next plague outbreak in Madagascar could be 's...,"[(265, 323), (1795, 1935), (149, 157), (1069, ..."
1,111111112,US bloggers banned from entering UK\n\nTwo pro...,"[(191, 219), (476, 556), (785, 798), (958, 101..."
2,111111113,Kate Steinle's death at the hands of a Mexican...,"[(1396, 1430), (3082, 3099), (3828, 3985), (36..."
3,111111114,U.S. judge frees Indonesian immigrant held by ...,"[(1705, 1824)]"
4,111111115,Here are all the sexual misconduct accusations...,"[(658, 700), (1870, 1893), (1655, 1745), (2389..."


In [6]:
#Convert list of tuples (of spans) to JSON string to easily be saved
semeval_si_cleaned = semeval_si_grouped.copy()
semeval_si_cleaned['propaganda_offsets'] = semeval_si_cleaned['propaganda_offsets'].apply(json.dumps)

In [ ]:
%%sql


In [7]:
#Save to processed data folder
output_path = BASE_DIR / "data" / "processed" / "semeval_si_cleaned.csv"
semeval_si_cleaned.to_csv(output_path, index=False)
print("SemEval SI DF is cleaned")

SemEval SI DF is cleaned


In [8]:
#Load SemEval technique classification data
semeval_tc = pd.read_csv(INTERIM_DIR / "semeval_task2_tc_merged.csv")
semeval_tc.head()

,article_id,technique,start_char,end_char,source_file,text_content
0,758756657,Repetition,5024,5036,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...
1,758756657,Repetition,5302,5314,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...
2,758756657,Loaded_Language,62,69,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...
3,758756657,Causal_Oversimplification,606,746,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...
4,758756657,Loaded_Language,4352,4361,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...


In [9]:
#Fill in missing techniques with 'Unknown'
semeval_tc['technique'] = semeval_tc['technique'].fillna('Unknown')

In [10]:
#Get unique techniques (one-hot encoding)
technique_dummies = semeval_tc['technique'].str.get_dummies(sep=',')
semeval_tc = pd.concat([semeval_tc, technique_dummies], axis=1)
semeval_tc.head()

,article_id,technique,start_char,end_char,source_file,text_content,Appeal_to_Authority,Appeal_to_fear-prejudice,Bandwagon,Black-and-White_Fallacy,...,Loaded_Language,Minimisation,Name_Calling,Red_Herring,Reductio_ad_hitlerum,Repetition,Slogans,Straw_Men,Thought-terminating_Cliches,Whataboutism
0,758756657,Repetition,5024,5036,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
1,758756657,Repetition,5302,5314,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,758756657,Loaded_Language,62,69,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
3,758756657,Causal_Oversimplification,606,746,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,758756657,Loaded_Language,4352,4361,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0


In [11]:
#Get relevant text being evaluated for each row (using span on text)
semeval_tc['span_text'] = semeval_tc.apply(lambda row: row['text_content'][row['start_char']:row['end_char']], axis=1)
semeval_tc.head()

,article_id,technique,start_char,end_char,source_file,text_content,Appeal_to_Authority,Appeal_to_fear-prejudice,Bandwagon,Black-and-White_Fallacy,...,Minimisation,Name_Calling,Red_Herring,Reductio_ad_hitlerum,Repetition,Slogans,Straw_Men,Thought-terminating_Cliches,Whataboutism,span_text
0,758756657,Repetition,5024,5036,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...,0,0,0,0,...,0,0,0,0,1,0,0,0,0,Islamization
1,758756657,Repetition,5302,5314,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...,0,0,0,0,...,0,0,0,0,1,0,0,0,0,Islamization
2,758756657,Loaded_Language,62,69,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...,0,0,0,0,...,0,0,0,0,0,0,0,0,0,outrage
3,758756657,Causal_Oversimplification,606,746,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...,0,0,0,0,...,0,0,0,0,0,0,0,0,0,"In order to convert to Islam, one says the sha..."
4,758756657,Loaded_Language,4352,4361,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...,0,0,0,0,...,0,0,0,0,0,0,0,0,0,egregious


In [12]:
#Each row should represent one unique span
aggregation_rules = {col: 'max' for col in technique_dummies.columns}
aggregation_rules.update({
    'article_id': 'first',
    'text_content': 'first',
    'start_char': 'first',
    'end_char': 'first',
    'span_text': 'first'
})
semeval_tc = semeval_tc.groupby(['article_id', 'start_char', 'end_char']).agg(aggregation_rules).reset_index(drop=True)
semeval_tc.head()

,Appeal_to_Authority,Appeal_to_fear-prejudice,Bandwagon,Black-and-White_Fallacy,Causal_Oversimplification,Doubt,Exaggeration,Flag-Waving,Labeling,Loaded_Language,...,Repetition,Slogans,Straw_Men,Thought-terminating_Cliches,Whataboutism,article_id,text_content,start_char,end_char,span_text
0,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,111111111,Next plague outbreak in Madagascar could be 's...,149,157,appeared
1,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,111111111,Next plague outbreak in Madagascar could be 's...,265,323,The next transmission could be more pronounced...
2,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,111111111,Next plague outbreak in Madagascar could be 's...,1069,1091,"a very, very different"
3,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,111111111,Next plague outbreak in Madagascar could be 's...,1334,1462,He also pointed to the presence of the pneumon...
4,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,111111111,Next plague outbreak in Madagascar could be 's...,1577,1616,but warned that the danger was not over


In [13]:
#Add some random examples of spans that are not propaganda because the span identification model may incorrectly label some
# #non-propaganda as propaganda, so the technique classifier model needs to be able to handle this
def get_negative_samples(si_df, num_samples=1500):
    neg_samples = []
    #Group SI data to know where the "forbidden" propaganda zones are
    grouped = si_df.groupby(['article_id', 'text_content'])

    for (article_id, text), group in grouped:
        if len(neg_samples) >= num_samples: break

        #Create a mask of the whole text: True = Propaganda, False = Clean
        is_propaganda = [False] * len(text)
        for _, row in group.iterrows():
            for i in range(int(row['start_char']), int(row['end_char'])):
                if i < len(is_propaganda): is_propaganda[i] = True

        #Find "Clean" blocks (sequences of False)
        clean_start = None
        for i, val in enumerate(is_propaganda):
            if not val and clean_start is None:
                clean_start = i
            elif val and clean_start is not None:
                if (i - clean_start) > 6: #Only take meaningful snippets (>6 chars)
                    neg_samples.append({
                        'article_id': article_id,
                        'start_char': clean_start,
                        'end_char': i,
                        'span_text': text[clean_start:i][:200], #Cap length to 200
                        **{col: 0 for col in technique_dummies}        #All techniques = 0
                    })
                clean_start = None
    return pd.DataFrame(neg_samples).sample(min(num_samples, len(neg_samples)))

print("Generating negative samples to reduce model bias")
df_tc_neg = get_negative_samples(semeval_si)
df_tc_neg.head()

Generating negative samples to reduce model bias


,article_id,start_char,end_char,span_text,Appeal_to_Authority,Appeal_to_fear-prejudice,Bandwagon,Black-and-White_Fallacy,Causal_Oversimplification,Doubt,...,Loaded_Language,Minimisation,Name_Calling,Red_Herring,Reductio_ad_hitlerum,Repetition,Slogans,Straw_Men,Thought-terminating_Cliches,Whataboutism
423,701299732,114,244,would spread from Madagascar have now taken h...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1355,729578579,14822,15614,by being lauded on the cover of the homosexua...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1225,728758697,1036,1232,he emphasized later.\nWhen asked about Farrak...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
528,703698295,2808,2818,in which,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
529,703698295,2850,3183,compete to defeat each other in the sycophanc...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [14]:
#Combine Positive (Propaganda) and Negative (Clean) examples
semeval_tc = pd.concat([semeval_tc, df_tc_neg], ignore_index=True).fillna(0)
semeval_tc.head()

,Appeal_to_Authority,Appeal_to_fear-prejudice,Bandwagon,Black-and-White_Fallacy,Causal_Oversimplification,Doubt,Exaggeration,Flag-Waving,Labeling,Loaded_Language,...,Repetition,Slogans,Straw_Men,Thought-terminating_Cliches,Whataboutism,article_id,text_content,start_char,end_char,span_text
0,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,111111111,Next plague outbreak in Madagascar could be 's...,149,157,appeared
1,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,111111111,Next plague outbreak in Madagascar could be 's...,265,323,The next transmission could be more pronounced...
2,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,111111111,Next plague outbreak in Madagascar could be 's...,1069,1091,"a very, very different"
3,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,111111111,Next plague outbreak in Madagascar could be 's...,1334,1462,He also pointed to the presence of the pneumon...
4,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,111111111,Next plague outbreak in Madagascar could be 's...,1577,1616,but warned that the danger was not over


In [15]:
def get_features(df):
    #Add Sentiment Score (-1.0 to 1.0): Propaganda often uses highly positive (Flag-Waving) or highly negative (Name-Calling) sentiment.
    df['sentiment'] = df['span_text'].apply(lambda x: TextBlob(str(x)).sentiment.polarity)

    #Add Punctuation Density: Excessive use of exclamation points or quotes often correlates with "Exaggeration" or "Doubt."
    df['punct_count'] = df['span_text'].apply(lambda x: len(re.findall(r'[!?"]', str(x))))

    #Add Lexical Diversity (Unique words / Total words): "Repetition" and "Slogans" have low lexical diversity.
    def lex_div(text):
        words = str(text).lower().split()
        if len(words) == 0: return 0
        return len(set(words)) / len(words)
    df['lexical_diversity'] = df['span_text'].apply(lex_div)

    #In the future, add Part-of-Speech (POS) Tags: High counts of adjectives and adverbs often signal "Loaded Language."

    return df

semeval_tc = get_features(semeval_tc)
semeval_tc.head()

,Appeal_to_Authority,Appeal_to_fear-prejudice,Bandwagon,Black-and-White_Fallacy,Causal_Oversimplification,Doubt,Exaggeration,Flag-Waving,Labeling,Loaded_Language,...,Thought-terminating_Cliches,Whataboutism,article_id,text_content,start_char,end_char,span_text,sentiment,punct_count,lexical_diversity
0,0,0,0,0,0,1,0,0,0,0,...,0,0,111111111,Next plague outbreak in Madagascar could be 's...,149,157,appeared,0.000000,0,1.000000
1,1,0,0,0,0,0,0,0,0,0,...,0,0,111111111,Next plague outbreak in Madagascar could be 's...,265,323,The next transmission could be more pronounced...,0.250000,0,1.000000
2,0,0,0,0,0,0,0,0,0,0,...,0,0,111111111,Next plague outbreak in Madagascar could be 's...,1069,1091,"a very, very different",0.000000,0,1.000000
3,0,1,0,0,0,0,0,0,0,0,...,0,0,111111111,Next plague outbreak in Madagascar could be 's...,1334,1462,He also pointed to the presence of the pneumon...,0.483333,0,0.863636
4,0,1,0,0,0,0,0,0,0,0,...,0,0,111111111,Next plague outbreak in Madagascar could be 's...,1577,1616,but warned that the danger was not over,0.000000,0,1.000000


In [16]:
#Reorder columns so that input columns come first, then engineered, then output columns
metadata_cols = ['article_id', 'text_content', 'span_text', 'start_char', 'end_char']
feature_cols  = ['sentiment', 'punct_count', 'lexical_diversity']
tech_cols = technique_dummies.columns.tolist()
final_column_order = (metadata_cols+ feature_cols + tech_cols)
semeval_tc = semeval_tc[final_column_order]
semeval_tc = semeval_tc.rename(columns={'start_char': 'start', 'end_char': 'end'})
semeval_tc.head()

,article_id,text_content,span_text,start,end,sentiment,punct_count,lexical_diversity,Appeal_to_Authority,Appeal_to_fear-prejudice,...,Loaded_Language,Minimisation,Name_Calling,Red_Herring,Reductio_ad_hitlerum,Repetition,Slogans,Straw_Men,Thought-terminating_Cliches,Whataboutism
0,111111111,Next plague outbreak in Madagascar could be 's...,appeared,149,157,0.000000,0,1.000000,0,0,...,0,0,0,0,0,0,0,0,0,0
1,111111111,Next plague outbreak in Madagascar could be 's...,The next transmission could be more pronounced...,265,323,0.250000,0,1.000000,1,0,...,0,0,0,0,0,0,0,0,0,0
2,111111111,Next plague outbreak in Madagascar could be 's...,"a very, very different",1069,1091,0.000000,0,1.000000,0,0,...,0,0,0,0,0,1,0,0,0,0
3,111111111,Next plague outbreak in Madagascar could be 's...,He also pointed to the presence of the pneumon...,1334,1462,0.483333,0,0.863636,0,1,...,0,0,0,0,0,0,0,0,0,0
4,111111111,Next plague outbreak in Madagascar could be 's...,but warned that the danger was not over,1577,1616,0.000000,0,1.000000,0,1,...,0,0,0,0,0,0,0,0,0,0


In [17]:
#Save to processed data folder
output_path = BASE_DIR / "data" / "processed" / "semeval_tc_cleaned.csv"
semeval_tc.to_csv(output_path, index=False)
print("SemEval TC DF is cleaned")

SemEval TC DF is cleaned
